### Define and Load Data
- you could start all the analysis with this cell!

In [1]:
import numpy as np
import sys
from pathlib import Path
sys.path.append(r'D:\Neural-Pipeline\source')
from analysis_utils.NeuralDataLoader import NeuralDataLoader, Dots3DMPConfig

subject = "zarya"
date = "20250411"
recording_depth = 15000

# Load session
loader = NeuralDataLoader()
loader.load_session(subject, date)

# Load all spike data for different alignments
stimOn_spikes = loader.get_spike_data(alignment='stimOn', good_units_only=True, good_trials_only=True)
saccOnset_spikes = loader.get_spike_data(alignment='saccOnset', good_units_only=True, good_trials_only=True)
postTargHold_spikes = loader.get_spike_data(alignment='postTargHold', good_units_only=True, good_trials_only=True)

# Load tuning data
tuning_spikes = loader.get_tuning_data(good_units_only=True, good_trials_only=True)

# Load behavioral data
behavior_dots3DMP = loader.get_behavioral_data(task='dots3DMP', good_trials_only=True)
behavior_tuning = loader.get_behavioral_data(task='tuning', good_trials_only=True)

# Load Unit Info
unit_info = loader.get_unit_info(good_units_only=True)
# MST_units = loader.get_units_by_area(unit_info, area_name='MST')
# VPS_units = loader.get_units_by_area(unit_info, area_name='VPS')
# dual_units = loader.get_units_by_area(unit_info, area_name='dual')
# Load configuration
config = Dots3DMPConfig(subject)

# Convert behavioral data
behavior_converted = config.convert_behavioral_data(behavior_dots3DMP, task='dots3DMP')
behavior_tuning_converted = config.convert_behavioral_data(behavior_tuning, task='tuning')

# Time Info
time_info = config.get_time_Info('dots3DMP')
time_info_tuning = config.get_time_Info('tuning')
time_axes_dots3DMP = config.get_time_axes('dots3DMP')

Loaded dots3DMP data: zarya20250411dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250411dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya


In [2]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from matplotlib.backends.backend_pdf import PdfPages
import os

depths = unit_info['depth']
real_depth = depths * (-1)

n_time_bins = tuning_spikes.shape[2]
trial_start = config.tuning_time_info['trial_start'] 
bin_size = config.tuning_time_info['bin_size']       
trial_stop = config.tuning_time_info['trial_stop']   
time_axis = trial_start + np.arange(n_time_bins) * bin_size

modality = behavior_tuning['modality']
ves_idx = np.where(modality == 1)[0]
vis_idx = np.where(modality == 2)[0]
com_idx = np.where(modality == 3)[0]
n_mod = 3

n_units, n_trials, n_time_bins = tuning_spikes.shape

depth_bin_size = 20
max_depth = int(np.max(depths))
depth_bins = np.arange(0, max_depth + depth_bin_size, depth_bin_size)
n_depth_bins = len(depth_bins)
slots_per_bin = 1

depth_sort_indices = np.argsort(real_depth, axis=0).flatten()
sorted_depths = depths[depth_sort_indices].flatten()

total_rows = n_depth_bins * slots_per_bin
z_sorted_spkrate = np.full((total_rows, n_time_bins, n_mod), np.nan)

depth_bin_to_base_row = {}
for i, depth_bin in enumerate(depth_bins):
    base_row = (n_depth_bins - 1 - i) * slots_per_bin
    depth_bin_to_base_row[depth_bin] = base_row

depth_bin_slot_counter = {depth_bin: 0 for depth_bin in depth_bins}

for i, unit_idx in enumerate(depth_sort_indices):
    unit_depth = sorted_depths[i]
    bin_idx = int(unit_depth // depth_bin_size) * depth_bin_size
    
    if bin_idx not in depth_bin_to_base_row:
        continue
    
    if depth_bin_slot_counter[bin_idx] >= slots_per_bin:
        continue
    
    unit_data = np.squeeze(tuning_spikes[unit_idx, :, :])
    baseline_mask = time_axis < 0
    
    mean_baseline = np.nanmean(unit_data[:, baseline_mask])
    std_baseline = np.nanstd(unit_data[:, baseline_mask])
    if std_baseline == 0 or np.isnan(std_baseline):
        std_baseline = 1e-6
    if mean_baseline == 0:
        mean_baseline = 1e-6
    unit_data_z = (unit_data - mean_baseline) / std_baseline
    
    base_row = depth_bin_to_base_row[bin_idx]
    slot = depth_bin_slot_counter[bin_idx]
    row_idx = base_row + slot
    
    for mod in range(n_mod):
        if mod == 0:
            unit_data_mod = unit_data_z[ves_idx, :]
        elif mod == 1:
            unit_data_mod = unit_data_z[vis_idx, :]
        else:
            unit_data_mod = unit_data_z[com_idx, :]
        
        mean_response_mod = np.mean(unit_data_mod, axis=0)
        z_sorted_spkrate[row_idx, :, mod] = mean_response_mod
    
    depth_bin_slot_counter[bin_idx] += 1

# Ensure the directory exists
output_dir = r"D:\Neural-Pipeline\results"
os.makedirs(output_dir, exist_ok=True)

pdf_filename = os.path.join(output_dir, f"{date}_dots3DMP_singleneuron_heatmap_poster.pdf")
with PdfPages(pdf_filename) as pdf:
    for mod in range(n_mod):
        plt.figure(figsize=(12, 10))
        
        sns.heatmap(
            z_sorted_spkrate[:, :, mod],
            cmap='RdBu_r',
            center=0,
            vmin=-1, vmax=1,
            cbar_kws={'label': 'Z-scored Firing Rate'},
            xticklabels=False,
            yticklabels=False)

        # Make event lines thicker and more visible
        stim_onset = np.where(time_axis >= 0)[0][0]
        plt.axvline(x=stim_onset, color='black', linestyle='-', linewidth=3)
        
        time_max = time_axis[-1]
        x_pos = np.where(time_axis >= (time_max - trial_stop))[0][0]
        plt.axvline(x=x_pos, color='darkgreen', linestyle='-', linewidth=3)
        
        # Bigger title with color
        mod_name = ['Vestibular', 'Visual', 'Combined']
        mod_colors = ['#E74C3C', '#3498DB', '#2ECC71']
        plt.title(f'{mod_name[mod]} Condition', fontsize=20, fontweight='bold', color=mod_colors[mod])

        plt.xlabel('Time (s)', fontsize=16, fontweight='bold')
        plt.ylabel('Depth (μm)', fontsize=16, fontweight='bold')

        # For labels at 0, 1, 2 seconds
        time_points = [0, 1, 2]  # The time points you want to label
        tick_indices = []
        tick_labels = []

        for t in time_points:
            # Find the closest index for each time point
            idx = np.argmin(np.abs(time_axis - t))
            tick_indices.append(idx)
            tick_labels.append(str(t))

        plt.xticks(tick_indices, tick_labels, fontsize=14)

        y_tick_indices = []
        y_tick_labels = []
        
        for depth_bin in sorted(depth_bins, reverse=True):
            base_row = depth_bin_to_base_row[depth_bin]
            
            # Make depth lines more visible
            if depth_bin != max(depth_bins):
                if depth_bin % 100 == 0:  # Major lines every 100 μm
                    plt.axhline(y=base_row, color='white', linestyle='-', alpha=0.5, linewidth=1.5)
                else:  # Minor lines
                    plt.axhline(y=base_row, color='white', linestyle='-', alpha=0.3, linewidth=0.5)
            
            # More frequent depth labels
            if depth_bin % 1000 == 0:  # Labels every 200 μm
                y_tick_indices.append(base_row + slots_per_bin // 2)
                y_tick_labels.append(f'{-recording_depth+depth_bin}')

        plt.yticks(y_tick_indices, y_tick_labels, fontsize=14)
        plt.tight_layout()

        png_filename = os.path.join(output_dir, f"{date}_dots3DMP_{mod_name[mod]}_heatmap_poster.png")
        # plt.savefig(png_filename, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved PNG: {png_filename}")
        
        pdf.savefig()
        plt.close()
print(f"Saved heatmaps to {pdf_filename}")

Saved PNG: D:\Neural-Pipeline\results\20250411_dots3DMP_Vestibular_heatmap_poster.png
Saved PNG: D:\Neural-Pipeline\results\20250411_dots3DMP_Visual_heatmap_poster.png
Saved PNG: D:\Neural-Pipeline\results\20250411_dots3DMP_Combined_heatmap_poster.png
Saved heatmaps to D:\Neural-Pipeline\results\20250411_dots3DMP_singleneuron_heatmap_poster.pdf


In [3]:
mean_baseline = np.mean(unit_data[:, baseline_mask])
std_baseline = np.std(unit_data[:, baseline_mask])
if std_baseline == 0:
    std_baseline = 1e-6
if mean_baseline == 0:
    mean_baseline = 1e-6
unit_data_z = (unit_data - mean_baseline) / std_baseline

base_row = depth_bin_to_base_row[bin_idx]
slot = depth_bin_slot_counter[bin_idx]
row_idx = base_row + slot

for mod in range(n_mod):
    if mod == 0:
        unit_data_mod = unit_data_z[ves_idx, :]
    elif mod == 1:
        unit_data_mod = unit_data_z[vis_idx, :]
    else:
        unit_data_mod = unit_data_z[com_idx, :]

    mean_response_mod = np.mean(unit_data_mod, axis=0)
    z_sorted_spkrate[row_idx, :, mod] = mean_response_mod
    
unit_data = np.squeeze(tuning_spikes[unit_idx, :, :])
baseline_mask = time_axis < 0

# Debug output
print(f"Unit {unit_idx}:")
print(f"  time_axis range: {time_axis[0]:.3f} to {time_axis[-1]:.3f}")
print(f"  baseline_mask True count: {np.sum(baseline_mask)}")
print(f"  unit_data shape: {unit_data.shape}")
print(f"  baseline data shape: {unit_data[:, baseline_mask].shape}")
print(f"  Contains NaN: {np.isnan(unit_data[:, baseline_mask]).any()}")

mean_baseline = np.mean(unit_data[:, baseline_mask])

Unit 0:
  time_axis range: -0.200 to 2.460
  baseline_mask True count: 10
  unit_data shape: (234, 134)
  baseline data shape: (234, 10)
  Contains NaN: False
